# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [6]:
!git clone -q https://github.com/rekhasreebandaru/flyrank-ml-internship.git
import pandas as pd

df = pd.read_csv("flyrank-ml-internship/data/raw/content_refresh_anonymized.csv")
print(df.shape)
for col in df.columns:
    print(col, "-", df[col].dtype)

fatal: destination path 'flyrank-ml-internship' already exists and is not an empty directory.
(30000, 44)
content_id - object
client_id - object
search_volume - float64
competition - float64
competition_level - object
cpc - float64
content_type - object
main_intent - object
word_count - float64
char_count - float64
provider_used - object
model_used - object
impressions_90d - int64
clicks_90d - int64
pageviews_90d - int64
sessions_90d - int64
users_90d - int64
engaged_sessions_90d - int64
ai_sessions_90d - int64
scroll_events_90d - int64
days_with_impressions - int64
days_with_sessions - int64
impressions_last_30d - int64
clicks_last_30d - int64
sessions_last_30d - int64
impressions_prev_30d - int64
clicks_prev_30d - int64
sessions_prev_30d - int64
content_age_days - int64
age_tier - object
age_tier_order - int64
days_since_last_update - int64
freshness_tier - object
word_count_tier - object
char_count_tier - object
ctr - float64
avg_position - float64
engagement_rate - float64
scroll_r

In [7]:
# Signal 1: Staleness vs decline
df["declined"] = (df["clicks_last_30d"] < df["clicks_prev_30d"]).astype(int)

bucket1 = df.groupby("freshness_tier").agg(
    n=("content_id", "count"),
    pct_declined=("declined", "mean")
).reset_index()
print("SIGNAL 1: Staleness (freshness_tier) vs decline in clicks")
print(bucket1)
print()

# Signal 2: CTR vs position
bucket2 = df.groupby("position_tier").agg(
    n=("content_id", "count"),
    avg_ctr=("ctr", "mean")
).reset_index()
print("SIGNAL 2: Position tier vs average CTR")
print(bucket2)

SIGNAL 1: Staleness (freshness_tier) vs decline in clicks
  freshness_tier      n  pct_declined
0           0-30  20480      0.208350
1           181+    174      0.091954
2          31-90    175      0.125714
3         91-180   9171      0.272707

SIGNAL 2: Position tier vs average CTR
  position_tier      n   avg_ctr
0          deep   1319  0.150212
1        page_1  11814  0.652467
2      page_3_5   7242  0.222484
3      striking   7304  0.323239
4         top_3   2321  1.483611


My rule: A page is worth prioritizing for action if it sits in a weak position tier (deep or page_3_5) but still gets meaningful search volume — meaning it has real potential that's being wasted by poor visibility. Staleness alone (days since update) is not a reliable signal on its own, since the bucket check showed no clean pattern between freshness and decline — so staleness only supports the rule when volume is also present, not as a standalone trigger.

Reason codes: weak_position_high_volume, weak_position_low_volume, strong_position_declining, stale_no_clear_pattern

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [8]:
# Score: weak position + high volume = high priority
weak_position = df["position_tier"].isin(["deep", "page_3_5"]).astype(int)
high_volume = (df["search_volume"] >= df["search_volume"].median()).astype(int)

df["score"] = weak_position * (1 + high_volume) * df["search_volume"]

def reason_code(row):
    if row["position_tier"] in ["deep", "page_3_5"] and row["search_volume"] >= df["search_volume"].median():
        return "weak_position_high_volume"
    elif row["position_tier"] in ["deep", "page_3_5"]:
        return "weak_position_low_volume"
    elif row["position_tier"] in ["top_3", "striking"]:
        return "strong_position_declining" if row["declined"] == 1 else "strong_position_stable"
    else:
        return "no_clear_signal"

df["reason_code"] = df.apply(reason_code, axis=1)

def action_label(row):
    if row["reason_code"] == "weak_position_high_volume":
        return "prioritize_refresh"
    elif row["reason_code"] == "weak_position_low_volume":
        return "monitor"
    elif row["reason_code"] == "strong_position_declining":
        return "investigate_decline"
    else:
        return "no_action"

df["action"] = df.apply(action_label, axis=1)

ranked = df.sort_values("score", ascending=False)

import os
os.makedirs("work/outputs", exist_ok=True)
ranked.to_csv("work/outputs/baseline_action_score.csv", index=False)

print("Rows written:", len(ranked))
print(ranked[["content_id", "position_tier", "search_volume", "score", "reason_code", "action"]].head(10))

Rows written: 30000
                 content_id position_tier  search_volume     score  \
12140  content_ef99c4abd9ab      page_3_5        74000.0  148000.0   
18701  content_deb54e9e19cd      page_3_5        60500.0  121000.0   
6972   content_bf67a444faef      page_3_5        60500.0  121000.0   
17907  content_5ec29ae79c60      page_3_5        60500.0  121000.0   
28282  content_454cc6654c6e      page_3_5        60500.0  121000.0   
8055   content_cd6760921db8      page_3_5        49500.0   99000.0   
16005  content_83e3da1394ac          deep        49500.0   99000.0   
22788  content_ee4630879d03      page_3_5        49500.0   99000.0   
2815   content_7868341d97dd      page_3_5        40500.0   81000.0   
8002   content_c841193dc692      page_3_5        40500.0   81000.0   

                     reason_code              action  
12140  weak_position_high_volume  prioritize_refresh  
18701  weak_position_high_volume  prioritize_refresh  
6972   weak_position_high_volume  prioritize

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [9]:
top10 = ranked.head(10)[["content_id", "position_tier", "search_volume", "score", "reason_code", "action"]]
print(top10.to_string(index=False))

          content_id position_tier  search_volume    score               reason_code             action
content_ef99c4abd9ab      page_3_5        74000.0 148000.0 weak_position_high_volume prioritize_refresh
content_deb54e9e19cd      page_3_5        60500.0 121000.0 weak_position_high_volume prioritize_refresh
content_bf67a444faef      page_3_5        60500.0 121000.0 weak_position_high_volume prioritize_refresh
content_5ec29ae79c60      page_3_5        60500.0 121000.0 weak_position_high_volume prioritize_refresh
content_454cc6654c6e      page_3_5        60500.0 121000.0 weak_position_high_volume prioritize_refresh
content_cd6760921db8      page_3_5        49500.0  99000.0 weak_position_high_volume prioritize_refresh
content_83e3da1394ac          deep        49500.0  99000.0 weak_position_high_volume prioritize_refresh
content_ee4630879d03      page_3_5        49500.0  99000.0 weak_position_high_volume prioritize_refresh
content_7868341d97dd      page_3_5        40500.0  81000.0 weak_

1. content_ef99c4abd9ab — Action: prioritize_refresh. Why: page_3_5 position with the highest search volume (74,000) in the dataset — real traffic being lost to weak placement. What would make it wrong: if this content's low ranking is intentional (e.g. deliberately deprioritized by the client) rather than a missed opportunity, refreshing it would waste effort.

2-5. (4 tied entries, all page_3_5, 60,500 volume, score 121,000) — Action: prioritize_refresh. Why: identical volume and position tier put these in a tie — the rule can't distinguish further within a tier. What would make it wrong: if these are near-duplicate content pieces competing with each other, "fixing" all four independently could be redundant work; a dedup check would help.

6-8. (3 tied entries, mixed deep/page_3_5, 49,500 volume) — Action: prioritize_refresh. Why: still high volume, though one shifts into "deep" position — a more severe visibility problem than page_3_5. What would make it wrong: "deep" position pages sometimes signal thin/low-quality content rather than a refresh opportunity — worth a quick manual check before committing resources.

9-10. (2 tied entries, page_3_5, 40,500 volume) — Action: prioritize_refresh. Why: lower volume than the top entries but still above median, so still flagged. What would make it wrong: this is close to the median threshold — a small change in the volume cutoff could exclude these, meaning the "high volume" label here is border-line, not clearly strong.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [10]:
# Weak pick check: how many top-10 entries are tied on identical score?
tie_check = ranked.head(10)["score"].value_counts()
print("Score tie counts in top 10:")
print(tie_check)
print()

# Leakage check: confirm no future-window or label-derived columns went into the score
score_inputs = ["position_tier", "search_volume"]
suspicious_cols = ["trend_direction", "trend_pct", "declined"]
print("Columns used in score:", score_inputs)
print("Confirmed NOT used in score:", suspicious_cols)
print("declined column exists only as a review/diagnostic signal for Signal 1 — never entered the score formula.")

Score tie counts in top 10:
score
121000.0    4
99000.0     3
81000.0     2
148000.0    1
Name: count, dtype: int64

Columns used in score: ['position_tier', 'search_volume']
Confirmed NOT used in score: ['trend_direction', 'trend_pct', 'declined']
declined column exists only as a review/diagnostic signal for Signal 1 — never entered the score formula.


Weak pick identified: 9 of the top 10 entries are tied across just 4 distinct score values (121,000 ×4, 99,000 ×3, 81,000 ×2). The rule can rank tiers correctly but cannot distinguish within a tier — it needs a tiebreaker (e.g. content_age_days or engagement_rate) to be genuinely useful at fine-grained ranking, not just coarse bucketing.

Leakage check: The score uses only position_tier and search_volume, both knowable independently of any outcome label. trend_direction, trend_pct, and the declined column (built for Signal 1's diagnostic check only) were never used in scoring — confirmed clean, no future-window or label-derived inputs.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.